In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# 1. Load Dataset (UCI Wine Quality Red)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=";")

# Feature Engineering to satisfy >= 12 features requirement
df["total_acidity"] = df["fixed acidity"] + df["volatile acidity"]
df["target"] = (df["quality"] >= 6).astype(int)  # Binary classification
df = df.drop(columns=["quality"])

X = df.drop(columns=["target"])
y = df["target"]

# 2. Train-Test Split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save test dataset (CSV) for Streamlit App upload requirement
test_data_export = pd.DataFrame(X_test, columns=X.columns)
test_data_export["target"] = y_test.values
test_data_export.to_csv("test_data.csv", index=False)
print("Saved test_data.csv successfully!")

# 3. Model Initialization
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "kNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Support Vector Machine": SVC(probability=True, random_state=42),    
}

# 4. Model Training, Metric Calculation & Export
metrics_summary = []
trained_models = {}

for name, model in models.items():
    if name in ["Logistic Regression", "kNN", "Support Vector Machine"]:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    metrics_summary.append(
        {
            "ML Model Name": name,
            "Accuracy": round(acc, 4),
            "AUC": round(auc, 4),
            "Precision": round(prec, 4),
            "Recall": round(rec, 4),
            "F1": round(f1, 4),
            "MCC": round(mcc, 4),
        }
    )

    # Save model and scaler artifacts
    joblib.dump(model, f'model_{name.lower().replace(" ", "_")}.joblib')

joblib.dump(scaler, "scaler.joblib")

# Print Evaluation Summary
results_df = pd.DataFrame(metrics_summary)
print("\n=== Model Evaluation Results ===")
print(results_df.to_string(index=False))

Saved test_data.csv successfully!


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



=== Model Evaluation Results ===
         ML Model Name  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.7406 0.8240     0.7683  0.7368 0.7522 0.4808
         Decision Tree    0.7469 0.7450     0.7586  0.7719 0.7652 0.4908
                   kNN    0.7250 0.8071     0.7456  0.7368 0.7412 0.4479
           Naive Bayes    0.7125 0.7852     0.7762  0.6491 0.7070 0.4358
Support Vector Machine    0.7594 0.8373     0.8013  0.7310 0.7645 0.5219
